# Olist Data Quality Audit

This notebook audits source quality and KPI reliability for the reporting layer.

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

repo_root = Path.cwd().parents[0] if Path.cwd().name == 'python' else Path.cwd()
processed = repo_root / 'data' / 'processed'
db_path = processed / 'olist_reporting.duckdb'

con = duckdb.connect(str(db_path))
con.execute((repo_root / 'sql' / '01_schema.sql').read_text())
con.execute((repo_root / 'sql' / '02_staging.sql').read_text())
con.execute((repo_root / 'sql' / '03_marts.sql').read_text())
print('Model refreshed:', db_path)

In [ ]:
dq = con.execute("""
SELECT *
FROM read_csv_auto('data/processed/data_quality_summary.csv', header=true)
ORDER BY issue_rows DESC
""").df()
dq

In [ ]:
join_risk = con.execute("""
WITH naive_join AS (
    SELECT SUM(oi.price + oi.freight_value) AS naive_gmv
    FROM staging.stg_order_items oi
    JOIN staging.stg_order_payments p ON oi.order_id = p.order_id
),
trusted AS (
    SELECT SUM(item_gmv) AS trusted_gmv FROM marts.fact_orders
)
SELECT
    naive_gmv,
    trusted_gmv,
    naive_gmv - trusted_gmv AS overstatement
FROM naive_join CROSS JOIN trusted
""").df()
join_risk

In [ ]:
delivery_review = con.execute("""
SELECT
    CASE
        WHEN delivery_days IS NULL THEN 'Not Delivered'
        WHEN delivery_days - estimated_delivery_days <= 0 THEN 'On time or early'
        WHEN delivery_days - estimated_delivery_days BETWEEN 1 AND 3 THEN '1-3 days late'
        WHEN delivery_days - estimated_delivery_days BETWEEN 4 AND 7 THEN '4-7 days late'
        ELSE '8+ days late'
    END AS delay_bucket,
    COUNT(*) AS orders,
    AVG(avg_review_score) AS avg_review_score
FROM marts.fact_orders
GROUP BY 1
ORDER BY 2 DESC
""").df()
delivery_review